# 10 · Sorting and Binary Search

*Standard library toolbox* · 9 exercises

Sort by any key, then binary-search both lists and answers.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ch-10) · [Answer key](https://bloomcode.github.io/cp/python/answer-key.html#ch-10)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import bisect
from functools import cmp_to_key

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 10.1 · Multi-key sort  `Core`

**Task.** Write `leaderboard(rows)` for `(name, score)` tuples: highest score first, ties in alphabetical order, using one `sorted` call with a tuple key.

**Example.** `leaderboard([("bo", 80), ("al", 95), ("cy", 80)])` -> `[('al', 95), ('bo', 80), ('cy', 80)]`

**Big O.** O(N log N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-10-1) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-10-1)

In [ ]:
# 10.1 Multi-key sort
def leaderboard(rows):
    raise NotImplementedError

In [ ]:
def test_ex_10_1():
    assert leaderboard([('bo', 80), ('al', 95), ('cy', 80)]) == [('al', 95), ('bo', 80), ('cy', 80)]


test_ex_10_1()
print('10.1 passed')

### 10.2 · Stability: sort twice  `Core`

**Task.** Write `by_dept_then_age(people)` for `(name, dept, age)` tuples: sort by age first, then stably by dept. Explain why the order of the two sorts matters.

**Example.** `[("a", "ops", 40), ("b", "dev", 30), ("c", "ops", 25)]` -> names `['b', 'c', 'a']`

**Big O.** O(N log N). Python's sort is stable: equal keys keep their existing order.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-10-2) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-10-2)

In [ ]:
# 10.2 Stability: sort twice
def by_dept_then_age(people):
    raise NotImplementedError

In [ ]:
def test_ex_10_2():
    assert [p[0] for p in by_dept_then_age([('a', 'ops', 40), ('b', 'dev', 30), ('c', 'ops', 25)])] == ['b', 'c', 'a']


test_ex_10_2()
print('10.2 passed')

### 10.3 · Custom comparison with cmp_to_key  `Practice`

**Task.** Write `largest_number(nums)`: arrange non-negative integers to form the largest number, as a string. Compare two numbers a, b by which of `str(a)+str(b)` or `str(b)+str(a)` is bigger.

**Example.** `largest_number([3, 30, 34, 5, 9])` -> `"9534330"`; `largest_number([0, 0])` -> `"0"`

**Big O.** O(N log N) comparisons.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-10-3) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-10-3)

In [ ]:
# 10.3 Custom comparison with cmp_to_key
def largest_number(nums):
    raise NotImplementedError

In [ ]:
def test_ex_10_3():
    assert largest_number([3, 30, 34, 5, 9]) == '9534330'
    assert largest_number([0, 0]) == '0'
    assert largest_number([10, 2]) == '210'


test_ex_10_3()
print('10.3 passed')

### 10.4 · Argsort  `Core`

**Task.** Write `argsort(arr)` returning the indices that would sort `arr`, using `sorted(range(len(arr)), key=arr.__getitem__)`.

**Example.** `argsort([30, 10, 20])` -> `[1, 2, 0]`

**Big O.** O(N log N).

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-10-4) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-10-4)

In [ ]:
# 10.4 Argsort
def argsort(arr):
    raise NotImplementedError

In [ ]:
def test_ex_10_4():
    assert argsort([30, 10, 20]) == [1, 2, 0]
    assert argsort([]) == []


test_ex_10_4()
print('10.4 passed')

### 10.5 · Count values in a range with bisect  `Core`

**Task.** Write `count_between(sorted_arr, lo, hi)` returning how many values fall in [lo, hi], using `bisect_left` and `bisect_right`.

**Example.** `count_between([1, 3, 3, 5, 8, 13], 3, 8)` -> `4`

**Big O.** O(log N) per query on a sorted list.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-10-5) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-10-5)

In [ ]:
# 10.5 Count values in a range with bisect
def count_between(sorted_arr, lo, hi):
    raise NotImplementedError

In [ ]:
def test_ex_10_5():
    assert count_between([1, 3, 3, 5, 8, 13], 3, 8) == 4
    assert count_between([1, 2], 5, 9) == 0


test_ex_10_5()
print('10.5 passed')

### 10.6 · Keep a list sorted with insort  `Core`

**Task.** Write `sorted_stream(values)` that inserts each value into a sorted list with `bisect.insort` and records the current median (lower middle for even sizes) after each insert.

**Example.** `sorted_stream([5, 2, 8, 1, 9])` -> `[5, 2, 5, 2, 5]`

**Big O.** O(log N) to find the spot, O(N) to shift, so O(N^2) total. Fine for N up to about 10^4.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-10-6) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-10-6)

In [ ]:
# 10.6 Keep a list sorted with insort
def sorted_stream(values):
    raise NotImplementedError

In [ ]:
def test_ex_10_6():
    assert sorted_stream([5, 2, 8, 1, 9]) == [5, 2, 5, 2, 5]


test_ex_10_6()
print('10.6 passed')

### 10.7 · Binary search on the answer  `Core`

**Task.** Write `min_speed(piles, hours)`: the smallest integer speed k such that eating `ceil(p / k)` hours per pile finishes within `hours`. Binary-search k between 1 and max(piles).

**Example.** `min_speed([3, 6, 7, 11], 8)` -> `4`; `min_speed([30, 11, 23, 4, 20], 5)` -> `30`

**Big O.** O(N log M) for M = max(piles). Search the answer space whenever "is k enough?" is easy to check and monotonic.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-10-7) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-10-7)

In [ ]:
# 10.7 Binary search on the answer
def min_speed(piles, hours):
    raise NotImplementedError

In [ ]:
def test_ex_10_7():
    assert min_speed([3, 6, 7, 11], 8) == 4
    assert min_speed([30, 11, 23, 4, 20], 5) == 30
    assert min_speed([30, 11, 23, 4, 20], 6) == 23


test_ex_10_7()
print('10.7 passed')

### 10.8 · A reusable first-true search  `Practice`

**Task.** Write `first_true(lo, hi, pred)` returning the smallest x in [lo, hi] with `pred(x)` true (pred goes False...False, True...True), or `hi + 1` if none. Use it for integer square root.

**Example.** `first_true(0, 100, lambda x: x * x >= 50)` -> `8`

**Big O.** O(log (hi - lo)) calls to pred.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-10-8) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-10-8)

In [ ]:
# 10.8 A reusable first-true search
def first_true(lo, hi, pred):
    raise NotImplementedError

In [ ]:
def test_ex_10_8():
    assert first_true(0, 100, lambda x: x * x >= 50) == 8
    assert first_true(0, 5, lambda x: False) == 6


test_ex_10_8()
print('10.8 passed')

### 10.9 · Coordinate compression  `Practice`

**Task.** Write `compress(arr)` replacing each value with its rank among the distinct values (0-based), using a sorted set and a dict.

**Example.** `compress([100, 5, 100, 42])` -> `[2, 0, 2, 1]`

**Big O.** O(N log N). Lets you index arrays by values up to 10^9 using only N slots.

[Workbook](https://bloomcode.github.io/cp/python/workbook.html#ex-10-9) · [Solution in the answer key](https://bloomcode.github.io/cp/python/answer-key.html#ex-10-9)

In [ ]:
# 10.9 Coordinate compression
def compress(arr):
    raise NotImplementedError

In [ ]:
def test_ex_10_9():
    assert compress([100, 5, 100, 42]) == [2, 0, 2, 1]


test_ex_10_9()
print('10.9 passed')